# Використання агентів разом з RAG

## Як працює звичайний конвеєр

Створимо звичайний конвеєр RAG.

In [1]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
gemini_client = genai.Client()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(index, gemini_client, instructions)

Ставимо питання.

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

/workspaces/LLM_Zoomcamp_2026/rag_helper.py:77: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = self.llm_client.interactions.create(


To run Ollama locally, follow these steps:

1.  **Install Ollama:** Visit [https://ollama.com/download](https://ollama.com/download) and download the installer for your operating system (macOS, Windows, or Linux). For Linux, you can run the command: `curl -fsSL https://ollama.com/install.sh | sh`.
2.  **Start the model:** Once installed, open a terminal and run `ollama run llama3`. This will download the model, start it locally, and open a chat interface.
3.  **Test the server:** You can verify the local server is running by executing `curl http://localhost:11434`.
4.  **Use Python:** To interact with Ollama via Python, install the client with `pip install ollama`. You can then use the `ollama.chat` function in your code to send prompts to the model.


А тепер допускаємо помилку в слові Ollama (пропускаємо букву "l").

In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The provided FAQ database does not contain information on how to run Ollama locally.


Модель не знаходить в базі потрібної для відповіді інформації і повідомляє нас про це. Отже, якщо результат пошуку буде незадовільний, звичайний конвеєр RAG дає збій. Модель не зможе дати відповідь на питання.

## Як працює конвеєр з агентом

### Крок 1. Визначення інструменту

Спочатку ми визначаємо функцію пошуку для моделі. Вона така сама, як в модулі rag_helper.py.

In [3]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Далі ми розповідаємо моделі про цю функцію. Модель не бачить нашого коду Python, лише схему, яка описує, що робить функція та які аргументи вона приймає. LLM не залежать від мови. Зрештою, ми просто здійснюємо HTTP-виклик, тому ми описуємо інструмент у JSON, а не в Python. Та сама схема працюватиме з TypeScript або Java.

In [4]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

Description - найважливіше поле, оскільки модель зчитує його, щоб вирішити, коли викликати функцію. Parameters - це схема JSON для аргументів, і ми позначаємо поле query як обов'язкове, щоб модель завжди заповнювала його.

### Крок 2. Надсилання запиту до моделі

Відправимо запит до моделі. Спочатку без використання інструментів.

In [5]:
message = "I just discovered the course. Can I join it?"

In [ ]:
interaction = gemini_client.interactions.create(
    model='gemini-3.1-flash-lite',
    input=message
)

interaction.output_text

/tmp/ipykernel_18698/3082309606.py:3: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = gemini_client.interactions.create(


'To give you an accurate answer, I need a little bit more information, as **I am an AI, not the administrator of your specific course.**\n\nTo figure out if you can still join, please check the following:\n\n### 1. Where did you find the course?\n*   **If it’s on a platform like Coursera, Udemy, edX, or LinkedIn Learning:** Yes, you can almost always join these at any time. They are usually self-paced.\n*   **If it’s a University or School course:** You will need to check the **"Add/Drop" deadline** in your student handbook or academic calendar. If the deadline has passed, you might need special permission from the professor or the registrar\'s office.\n*   **If it’s a private workshop or cohort-based course:** Check their website for an "Enrollment" button. If it says "Waitlist" or "Closed," you may have to wait for the next session.\n\n### 2. How to find out for sure:\n*   **Check the Syllabus or FAQ:** Look for a section labeled "Enrollment," "Registration," or "Timeline."\n*   **Ch

Ми отримали дуже загальні рекомендації, що цілком зрозуміло. Тепер запитаємо модель про теж саме, але використовуючи інструменти.

In [7]:
interaction = gemini_client.interactions.create(
    model='gemini-3.1-flash-lite',
    input=message,
    tools=[search_tool]
)

fc_step = next(s for s in interaction.steps if s.type == "function_call")

В коді вище генераторний вираз проходить по списку interaction.steps (де розміщені всі кроки, згенеровані моделлю) та фільтрує їх за умовою if s.type == "function_call", повертаючи лише ті кроки, де модель запросила виклик зовнішньої функції.

Вбудована функція next() бере найперший елемент, який згенерував наш вираз (тобто перший підхожий step). Як тільки перший потрібний елемент знайдено, обчислення зупиняються — next() не витрачає час на перегляд решти списку.

У змінну fc_step записується об'єкт кроку виклику функції. Далі з нього можна легко дістати назву функції (fc_step.name) та аргументи (fc_step.arguments), які сформувала Gemini.

Надрукуємо відповідь моделі.

In [8]:
print("Текст відповіді:", interaction.output_text)
print("Функціональний виклик:", fc_step)

Текст відповіді: 
Функціональний виклик: FunctionCallStep(id='J5CTk5uu', arguments={'query': 'can I join the course late'}, name='search', type='function_call', signature='EjQKMgERTTIPYqZIrPnBldjHMTC6De4J+o9cJidht/GKOlpoc/sr9ukxUcLfLqp3uH64TFL+')


Бачимо, що текстової відповіді немає. Отже модель вирішила, що перш ніж дати відповідь, їй потрібно пошукати в розділі поширених запитань. Замість відповіді вона просить нас запустити функцію пошуку.

Звернімо увагу на аргументи. Модель не передала наше питання дослівно. Вона вирішила, що необроблене запитання не найкращий запит для пошуку. Тому вона переписала наше запитання щоб воно краще відповідало ключовим словам. Ось так: «can I join the course late».

### Крок 3. Виклик функції

Викликаємо функцію, використовуючи надані моделлю аргументи. Результат її роботи ми перетворюємо у формат json. В ньому треба буде передавати їх моделі вдруге. Також у json можна красиво вивести результати на екран.  

In [11]:
import json

if fc_step.name == "search":
    result = search(**fc_step.arguments)
    result_json = json.dumps(result, indent=2)
    print(f"Function execution result: {result_json}")

Function execution result: [
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them."
  },
  {
    "id": "977bf7786c",
    

### Повторний запит до моделі

Ми повертаємо моделі результати пошуку. Питання не повторюємо. Воно збережено в об'єкті попереднього запиту interaction. 

In [12]:
final_interaction = gemini_client.interactions.create(
    model='gemini-3.1-flash-lite',
    input=[
        {
            "type": "function_result",
            "name": fc_step.name,
            "call_id": fc_step.id,
            "result": [{"type": "text", "text": result_json}],
        }
    ],
    tools=[search_tool],
    previous_interaction_id=interaction.id
)

print(final_interaction.output_text)

Yes, you can join the course at any time!

You don't need a formal registration to begin. You can start working through the materials, watching the videos, and exploring the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp) whenever you are ready.

**A few things to keep in mind regarding your progress:**

*   **Learning:** You can work through the course content at your own pace.
*   **Homework:** You can submit homework assignments through the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) as long as the submission forms are open.
*   **Certificates:** If your goal is to earn a certificate, please note that you must finish a capstone project and complete the required peer reviews during a "live" cohort period. You cannot earn a certificate in self-paced mode.

To get started, check out the [LLM Zoomcamp documentation](https://datatalks.club/docs/courses/llm-zoomcamp/) and the general [Zoomcamp logistics](https://datatalks.club/docs/co

## Агентський цикл

У попередньому розділі ми відправили повідомлення, отримали від моделі запит на виклик функції, викликали її, отримали результат і відправили його до моделі з новим запитом. 

Це працює для одного виклику функції. Але ламається, коли модель хоче виконати пошук кілька разів або коли перший пошук не дає релевантний результат. Ми заздалегідь не знаємо, скільки викликів знадобиться моделі. Тому нам потрібен цикл, який постійно викликає модель і запускає інструменти, доки процес не буде завершено. Агент — це саме те, що потрібно.

### Анатомія агента

Агент складається з трьох частин:

- Інструкції, роль та поведінка, яких ми від нього хочемо. Ми передаємо це як developer-повідомлення. Чим кращі інструкції, тим краще допомагає агент.
- Інструменти, функції, які агент може викликати для виконання завдання. Для нас це лише search.
- Пам'ять, історія повідомлень. Ми додаємо в новий запит кожен попередній запит, кожен вивід моделі та кожен результат інструменту. Агент зчитує це, щоб знати, що він уже спробував.

### Системна інструкція для моделі

Досі ми покладалися на модель, щоб визначити, коли саме вдаватись до пошуку. Ми зробимо це надійнішим за допомогою developer-повідомлення, яке пояснює моделі, як їй поводитись. Саме тут ми призначаєму агенту його роль. Те саме повідомлення також підштовхує його до кількох пошуків. Тож ми можемо спостерігати за виконанням циклу більше одного разу.

In [5]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

#### Обгортка для функції пошуку

Ми неодноразово будемо викликати функцію пошуку. Тому є сенс створити допоміжну функцію-обгортку, що буде викликати функцію пошуку, серіалізовувати результат і повертати його у такому вигляді, як очікує його АРІ.

In [6]:
import json

def make_call(call):

    if call.name == "search":
        result = search(**call.arguments)

    result_json = json.dumps(result, indent=2)

    return [{
        "type": "function_result",
        "name": call.name,
        "call_id": call.id,
        "result": [{"type": "text", "text": result_json}],
    }]

#### Цикл агента

Створимо цикл агента та обгорнемо його в функцію для зручности багаторазового використання. Ця функція приймає інструкції та питання як параметри та повертає остаточну відповідь.

In [7]:
def agent_loop(instructions, question, model='gemini-3.1-flash-lite', max_iterations=5) -> str:
    """
    This function implements an agent loop that interacts with the Gemini model to answer a question based on provided instructions. It continues to make function calls and process responses until no further function calls are made or the maximum number of iterations is reached."""
    iter = 1

    while True:
        print(f"\nIteration {iter}:")
        has_function_calls = False

        interaction = gemini_client.interactions.create(
            model=model,
            system_instruction=instructions,
            input=question,
            tools=[search_tool],
            previous_interaction_id=interaction.id if iter > 1 else None
        )

        for step in interaction.steps:
            if step.type == "function_call":
                print(f"Interaction ID: {interaction.id}") # For debugging purposes
                print(f"Function call: {step.name} with arguments: {step.arguments}")
                question = make_call(step)
                has_function_calls = True
            elif hasattr(step, "content") and step.content:
                for part in step.content:
                    if hasattr(part, "text"):
                        print("ASSISTANT:")
                        last_answer = part.text 
                        print(part.text)

        if has_function_calls == False:
            break

        iter += 1
        if iter > max_iterations:
            print("Maximum iterations reached.")
            break
        
    return last_answer    

Спробуємо викликати нашу функцію. Спочатку повторимо запитання про початок курсу.

In [11]:
print(agent_loop(instructions, "I just discovered the course. Can I still join it?"))


Iteration 1:
Interaction ID: v1_ChdCY0JrYXJmM0hiUGpuc0VQbU5xUHNRYxIXQmNCa2FyZjNIYlBqbnNFUG1OcVBzUWM
Function call: search with arguments: {'query': 'can I join the course late'}

Iteration 2:
ASSISTANT:
Yes, you can absolutely still join the course!

You are free to start learning whenever you want, as all the videos and materials are available on GitHub and the course website.

Here are a few important things to keep in mind:

*   **Certificate:** If your goal is to earn a certificate, you must submit your final capstone project and complete the required peer reviews while the current "live" cohort is accepting them.
*   **Workflow:** You can follow the course materials, watch the videos, and go through the notebooks at your own pace.
*   **Homework:** You are welcome to work through the homework assignments, but please note that for homework submissions, you must meet the deadlines listed in the course management platform.

Are there any other areas of the course—such as the technic

Спробуємо задати питання про Ollama з помилкою.

In [9]:
print(agent_loop(instructions, "How do I run Olama locally?"))


Iteration 1:
Interaction ID: v1_ChdOcjVrYXFUc0xxVGR2ZElQdnRqSTJRcxIXTnI1a2FxVHNMcVRkdmRJUHZ0akkyUXM
Function call: search with arguments: {'query': 'how to run Olama locally'}

Iteration 2:
ASSISTANT:
To run Ollama locally, follow these steps:

### 1. Installation
Visit [https://ollama.com/download](https://ollama.com/download) and download the version for your operating system:

*   **macOS**: Download and install the `.pkg` file.
*   **Windows**: Download and install the `.msi` file.
*   **Linux**: Run this command in your terminal:
    ```bash
    curl -fsSL https://ollama.com/install.sh | sh
    ```

### 2. Verify and Run
Once installed, open your terminal and run a model (e.g., LLaMA 3) to start a chat interface:
```bash
ollama run llama3
```
*This command will download the model (~4GB) and start it locally.*

### 3. Check the Server
To confirm the Ollama server is running correctly, run:
```bash
curl http://localhost:11434
```
You should receive a response indicating the server 

Тут виникає невелика проблема. Модель часто відповідає після першого пошуку, навіть коли допомогли б додаткові пошуки. Вона міркує, що вже знає достатньо, тож навіщо турбуватися. Ми підштовхуємо її дослідити більше, переписуючи інструкції.

In [12]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [13]:
agent_loop(instructions, "I just discovered the course. Can I join it?")


Iteration 1:
Interaction ID: v1_ChdFOEprYXJ2bE5fdVEyOG9QcnNfc3FBdxIXRThKa2FydmxOX3VRMjhvUHJzX3NxQXc
Function call: search with arguments: {'query': 'can I join the course late registration enrollment'}

Iteration 2:
ASSISTANT:
Yes, you can absolutely join the course!

You are free to start learning at any time, as the course materials (videos, notebooks, and GitHub repositories) remain available. You do not need to wait for a specific confirmation to begin.

However, there are a few important things to keep in mind regarding certificates and deadlines:

*   **Certificates:** You can only receive a certificate if you participate with a "live" cohort. While you can work through the materials and prepare your capstone project in self-paced mode, the final project submission and required peer reviews must happen while a live cohort is still accepting them.
*   **Homework:** You can follow the lessons and complete the homework, but please check the [course management platform](https://cour

'Yes, you can absolutely join the course!\n\nYou are free to start learning at any time, as the course materials (videos, notebooks, and GitHub repositories) remain available. You do not need to wait for a specific confirmation to begin.\n\nHowever, there are a few important things to keep in mind regarding certificates and deadlines:\n\n*   **Certificates:** You can only receive a certificate if you participate with a "live" cohort. While you can work through the materials and prepare your capstone project in self-paced mode, the final project submission and required peer reviews must happen while a live cohort is still accepting them.\n*   **Homework:** You can follow the lessons and complete the homework, but please check the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) to see if the submission forms for previous weeks are still open.\n*   **Getting Started:** A good place to begin is the [LLM Zoomcamp GitHub repository](https://github.com/DataTalk

Модель все одно не забажала робити більше пошуків. Вона може чинити на свій розсуд. Тому не варто очікувати, що вона буде дотримуватись інструкцій кожного разу.

І ще один момент... Нам треба, щоб модель відповідала тільки на питання по курсу, а не на будь які. Для цього ще трохи змінимо системний промпт.

In [14]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

Та спробуємо поставити "ліве" питання.

In [16]:
print(agent_loop(instructions, "what's queen gambit?"))


Iteration 1:
Interaction ID: v1_Chc1TU5rYXBPVEg1aWt4TjhQekp6XzhBcxIXNU1Oa2FwT1RINWlreE44UHpKel84QXM
Function call: search with arguments: {'query': 'queen gambit'}

Iteration 2:
ASSISTANT:
I'm sorry, but I cannot find any information regarding "Queen's Gambit" in the course materials. It appears that this topic may be off-topic for this course.

Are there any other areas related to the course logistics or content that you would like to explore?
I'm sorry, but I cannot find any information regarding "Queen's Gambit" in the course materials. It appears that this topic may be off-topic for this course.

Are there any other areas related to the course logistics or content that you would like to explore?
